In [3]:
import sys
import os
import h5py
import numpy as np
import torch
from sklearn.metrics import median_absolute_error
from dtaidistance import dtw
import scipy.stats


# Add src/ directory to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))

In [ ]:
from lora import LoRALinear
from processor import LLMTIMEPreprocessor
from data import run_forecast
from qwen import load_qwen

/Users/saamnazem/M2_stuff/M2_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/saamnazem/M2_stuff/M2_venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Trained (best) Metrics of Performance on System IDs

In [7]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd

systems = [300,900,814,584,900,884,869,516,560,573]

preprocessor = LLMTIMEPreprocessor()
trajectories = h5py.File("../lotka_volterra_data.h5", "r")["trajectories"][:]

# --- Store results
metrics = {
    "System": [],
    "MSE": [],
    "MAE": [],
    "R2 Score": [],
    "MedAE": [],
    "DTW": [],
    "Pearson": []
}

device = torch.device("cpu")
model, tokenizer = load_qwen()
model.to(device)

# Inject LoRA layers
lora_rank = 4
for layer in model.model.layers:
    layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj, r=lora_rank)
    layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj, r=lora_rank)

# Load trained LoRA weights
model.load_state_dict(torch.load("../models/lora_qwen2.5_final_x2.pth", map_location="cpu"), strict=False)
model.eval()
idx = 1

for system_id in systems:
    # Load sequence and run forecast
    sequence = trajectories[system_id, :, :]
    input_sequence = sequence[:60]
    true_output_sequence = sequence[60:]

    preprocessed_input = preprocessor.preprocess_sequence(input_sequence)
    tokenized_input = preprocessor.tokenize_sequence(preprocessed_input)
    input_tensor = torch.tensor([tokenized_input]).to(model.device)

    prediction = run_prediction(model, tokenizer, input_tensor, 1)

    # Clip length to match in case decoded output is shorter
    n_pred = min(len(true_output_sequence), len(prediction))
    y_true = true_output_sequence[:n_pred].flatten()
    y_pred = prediction[:n_pred].flatten()

    # Compute metrics
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    dtw_score = dtw.distance(y_true, y_pred)
    pearson = scipy.stats.pearsonr(y_true, y_pred)[0]

    metrics["System"].append(idx)
    metrics["MSE"].append(mse)
    metrics["MAE"].append(mae)
    metrics["R2 Score"].append(r2)
    metrics["MedAE"].append(medae)
    metrics["DTW"].append(dtw_score)
    metrics["Pearson"].append(pearson)
    
    idx+=1

# --- Convert to DataFrame and display
metrics_df = pd.DataFrame(metrics)
print(metrics_df)

# Optional: Save to CSV
metrics_df.to_csv("../csv/metrics_FINALFINAL.csv", index=False)

/Users/saamnazem/M2_stuff/M2_venv/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/saamnazem/M2_stuff/M2_venv/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/Users/saamnazem/M2_stuff/M2_venv/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/Users/saamnazem/M2_stuff/M2_venv/l

   System       MSE       MAE  R2 Score     MedAE       DTW   Pearson
0       1  0.023936  0.115251  0.903528  0.087113  0.893156  0.956013
1       1  0.014284  0.071380  0.793386  0.044355  0.903033  0.942218
2       1  0.005249  0.056619  0.990960  0.040131  0.637583  0.997974
3       1  0.043518  0.150483  0.851242  0.115788  1.048611  0.927711
4       1  0.014284  0.071380  0.793386  0.044355  0.903033  0.942218
5       1  0.020776  0.110883  0.770441  0.089703  0.600591  0.904748
6       1  0.013536  0.091715  0.765534  0.071264  0.681897  0.891696
7       1  0.278208  0.338082  0.811387  0.218586  3.333431  0.955676
8       1  0.288848  0.369850  0.855647  0.191678  2.006477  0.933991
9       1  0.005494  0.060681  0.967776  0.054831  0.620349  0.985357


### CTX Metrics of Performance on System IDs

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import torch
import numpy as np

# --- Store results
metrics = {
    "System ID": [],
    "Context Length": [],
    "MSE": [],
    "MAE": [],
    "R2 Score": []
}

device = torch.device("cpu")
systems = [900, 920, 940, 950, 970]
ctx_model_name = {
    128: "../models/lora_qwen2.5_ctx128_testing.pth",
    512: "../models/lora_qwen2.5_ctx512_testing.pth",
    768: "../models/lora_qwen2.5_ctx768_testing.pth"
}

# Load shared components
model_base, tokenizer = load_qwen()
model_base.to(device)

preprocessor = LLMTIMEPreprocessor()
trajectories = h5py.File("../lotka_volterra_data.h5", "r")["trajectories"][:]

def run_prediction(model, tokenizer, input_tensor):
    attention_mask = (input_tensor != tokenizer.pad_token_id).long()
    with torch.no_grad():
        output_tokens = model.generate(
            input_tensor, 
            attention_mask=attention_mask,
            max_new_tokens=1000,
            min_length=1000
        )
    generated_tokens = output_tokens[0].tolist()[len(input_tensor[0]):]
    decoded_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    try:
        parsed = np.array([
            list(map(float, pair.split(",")))
            for pair in decoded_output.split(";") if "," in pair
        ])
        return parsed[:40] * 10 if parsed.shape[0] >= 40 else np.zeros((40, 2))
    except ValueError:
        return np.zeros((40, 2))

# Evaluate each model
for ctx_length, model_path in ctx_model_name.items():
    # Reload base model each time for clean injection
    model, _ = load_qwen()
    model.to(device)
    model.eval()

    # Inject LoRA
    for layer in model.model.layers:
        layer.self_attn.q_proj = LoRALinear(layer.self_attn.q_proj, r=8)
        layer.self_attn.v_proj = LoRALinear(layer.self_attn.v_proj, r=8)

    model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=False)
    model.eval()

    for system_id in systems:
        sequence = trajectories[system_id, :, :]
        input_sequence = sequence[:60]
        true_output_sequence = sequence[60:]

        preprocessed_input = preprocessor.preprocess_sequence(input_sequence)
        tokenized_input = preprocessor.tokenize_sequence(preprocessed_input)
        input_tensor = torch.tensor([tokenized_input]).to(model.device)

        prediction = run_prediction(model, tokenizer, input_tensor)

        n_pred = min(len(true_output_sequence), len(prediction))
        y_true = true_output_sequence[:n_pred].flatten()
        y_pred = prediction[:n_pred].flatten()

        # Metrics
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        metrics["System ID"].append(system_id)
        metrics["Context Length"].append(ctx_length)
        metrics["MSE"].append(mse)
        metrics["MAE"].append(mae)
        metrics["R2 Score"].append(r2)

# --- Save Results
metrics_df = pd.DataFrame(metrics)
print(metrics_df)
metrics_df.to_csv("../csv/forecast_metrics_by_context.csv", index=False)


    System ID  Context Length         MSE        MAE   R2 Score
0         900             128    0.326417   0.507231  -3.721501
1         920             128   14.279890   3.121750  -2.149117
2         940             128    5.138399   1.789814  -7.973430
3         950             128    0.453862   0.443624   0.784901
4         970             128    0.056366   0.180561  -0.985601
5         900             512    0.038370   0.149452   0.445000
6         920             512    0.530201   0.658000   0.883076
7         940             512    6.691081   1.994684 -10.684953
8         950             512  158.452394  10.700654 -74.095219
9         970             512    0.056366   0.180561  -0.985601
10        900             768    0.122484   0.287165  -0.771690
11        920             768   14.279890   3.121750  -2.149117
12        940             768   15.821130   2.833434 -26.629192
13        950             768    4.098806   1.410243  -0.942544
14        970             768    0.05636